In [ ]:
import pandas as pd
df = pd.read_csv("insurance.csv")
df.head()

Pipeline : -->
Pipeline is process of sequence steps that preprocess data and passes that data through pipeline without repeatating every task manually and give predictions

In [ ]:
import pandas as pd

X_train = pd.DataFrame({
    "region": [
        "north", "south", "west",
        "north", "south", "west",
        "north", "south", "west"
    ]
})

y_train = pd.Series([
    100, 120, 150,
    110, 130, 160,
    105, 125, 155
])

X_test = pd.DataFrame({
    "region": ["north", "south", "west"]
})

y_test = pd.Series([108, 128, 158])

In [ ]:
## Importing required lib for ML
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

model = Pipeline(
    steps = [
        ("encoder", OneHotEncoder(handle_unknown="ignore",sparse_output=False)),
        ("regressor", LinearRegression())
    ]
)
model.fit(X_train,y_train)
prediction = model.predict(X_test)

print(prediction)

mae = mean_absolute_error(y_test,prediction)
print("MAE: ",mae)

print(X_test.shape)
print(prediction.shape)

ColumnTransformer :--> It helps us to put which column need encoding if data in text. and for numerical data we will passthrough . basically it helps to encode only columns that is we need to encode

Data Leakage :--> Data leakage in machine learning is similar: information that shouldn't be available during model training or evaluation influences the process, making the results misleading.

SimpleImputer
An imputer fills missing values using a chosen strategy.

In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer

df = pd.DataFrame({
    "salary": [30000, None, 40000, 50000]
})

# Create the imputer
imputer = SimpleImputer(
    strategy = "most_frequent"
)

# Fill the missing salary
df[["salary"]] = imputer.fit_transform(df[["salary"]])

print(df)

In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer

X_train = pd.DataFrame({
    "amount": [10, 20, None, 30]
})
X_test = pd.DataFrame({
    "amount": [100, None]
})

# Create the imputer
imputer = SimpleImputer(
    strategy = "median"
)

# Learn from training data and fill its missing value
X_train_imputer = imputer.fit_transform(
    X_train
)

# Use the same learned median on test data
X_test_imputer = imputer.transform(
    X_test
)
print("X_train: ",X_train_imputer)
print("X_test: ",X_test_imputer)

SimpleImputer inside a Pipeline
magine we're predicting salary using years of experience, but one value is missing.
Experience (years)	Salary (₹ thousands)
1	30
2	40
Missing	50
4	70
5	80

In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

X_train = pd.DataFrame({
    "experience": [1, 2, None, 4, 5]
})

y_train = pd.Series([30, 40, 50, 70, 80])

X_test = pd.DataFrame({
    "experience": [3, None, 6]
})

model = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy = "median")),
        ("regressor",LinearRegression())
    ]
)
model.fit(X_train,y_train)
prediction = model.predict(X_test)

print(prediction)
     

Categorical preprocessing Pipeline
We can use two operations in sequence:
1. SimpleImputer — fill the missing city with the most frequent city in the training data.
2. OneHotEncoder — convert the completed city column into numerical columns.
    
Raw categorical data

Pune, Mumbai, Missing, Pune, Nashik

SimpleImputer

Replace the missing city with Pune

OneHotEncoder

Create numerical indicator columns

Numerical features ready for the model

In [ ]:
import pandas as pd

X_train = pd.DataFrame({
    "city": ["Pune", "Mumbai", None, "Pune", "Nashik"]
})

X_test = pd.DataFrame({
    "city": ["Mumbai", None]
})

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

categorical_pipeline = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="most_frequent")),
        ("encoder",OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)
X_train_encoder = categorical_pipeline.fit_transform(X_train)
print(X_train_encoder)

print(
    categorical_pipeline
    .named_steps["encoder"]
    .get_feature_names_out(["city"])
)

X_test_encoder = categorical_pipeline.transform(X_test)
print(X_test_encoder)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

X_train = pd.DataFrame({
    "payment": ["Card", "Cash", None, "Card", "UPI"]
})

categorical_Pipeline = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="most_frequent")),
        ("encoder",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
    ]
)
pipeline = categorical_Pipeline.fit_transform(X_train)
print(pipeline)
print(categorical_Pipeline.named_steps["encoder"].get_feature_names_out(["payment"]))

Input features (X)

Age, BMI, Smoker, Region

ColumnTransformer

Routes each column group to its own preprocessing workflow

Numerical Pipeline

Age, BMI

SimpleImputer

Median imputation


Categorical Pipeline

Smoker, Region

SimpleImputer

OneHotEncoder



Prepared features → Regression model


Numerical + categorical preprocessing

In [ ]:
##Importing the needed Libreries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error

df = pd.DataFrame({
    "age": [25, 40, 35, 50, 28, 45, 32, 55, 38, 29, 48, 34],
    "bmi": [22.5, None, 27.1, 31.0, 24.3, 29.5,
            26.0, 33.2, 28.1, 23.5, 32.0, 25.7],
    "smoker": ["yes", "no", None, "no", "no", "yes",
               "no", "yes", "no", "no", "yes", "no"],
    "region": ["north", "south", "west", "east", "north", "south",
               "west", "east", "south", "north", "west", "east"],
    "charges": [18000, 12000, 25000, 14000, 10000, 27000,
                11000, 30000, 13000, 9000, 28000, 11500]
})

## 
X = df.drop(columns=["charges"])
y = df["charges"]

## Splitong the datastes. suing train_test_split
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## Creating variables for numerical and categorical features
numerical_features = ["age","bmi"]
categorical_features = ["region","smoker"]


## Creating a Pipeline to handel numerical missing values
numerical_pipeline = Pipeline(
    steps=[
        ("impute",SimpleImputer(strategy = "median"))
    ]
)

## Creating a Pipeline to handel categorical missing values
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
    ]
)

pipeline = categorical_pipeline.fit_transform(X_train)
print(pipeline)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric",numerical_pipeline,numerical_features),
        ("categorical",categorical_pipeline,categorical_features)
    ]
)
model = Pipeline(
    steps=[
    ("preprocessor",preprocessor),
    ("model",LinearRegression())
    ]
)

model.fit(X_train,y_train)
prediction = model.predict(X_test)

mae = mean_absolute_error(y_test,prediction)
print("MAE: ",mae)

In [ ]:
results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": prediction
})

results["Absolute Error"] = (
    results["Actual"] - results["Predicted"]
).abs()

print(results.round(2))

Feature Scaling meand Both features are transformed to a more comparable numerical scale. Their original units are no longer retained in the scaled values.
Their numerical scales are very different. Feature scaling transforms numerical features so they're represented on a more comparable scale

Some machine learning algorithms are sensitive to the scale of their input features.
Feature scaling can help these algorithms treat features more comparably.

It's especially useful for algorithms such as:
- K-Nearest Neighbors (KNN)
- Support Vector Machines (SVM)
- Many gradient-based models, including neural networks
- Regularized linear models such as Ridge and Lasso
Important: Scaling isn't required for every model. Decision Trees, Random Forests, and Gradient Boosting Trees generally don't need it. Ordinary Linear Regression also generally works without scaling, though scaling can be helpful in some situations.

StandardScaler

Transforms numerical features so they have approximately a mean of 0 and a standard deviation of 1, based on the data used to fit the scaler.

Often a good starting point for models sensitive to feature scales.


MinMaxScaler

Usually scales feature values to a chosen range, commonly 0 to 1.

Useful when a specific numerical range is desirable, though extreme values can affect the result.

In [ ]:
from sklearn.preprocessing import StandardScaler

numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [ ]:
## Standard Scaler
import pandas as pd 
from sklearn.preprocessing import StandardScaler

df = pd.DataFrame({
    "age": [20, 30, 40]
})

scaler = StandardScaler()

Data = scaler.fit_transform(df)
print(Data)

In [ ]:
df1 = pd.DataFrame({
    "salary": [20000, 50000, 80000]
})

scaler = StandardScaler()
DAta = scaler.fit_transform(df1)
print(DAta)

In [ ]:
## MIN MAX SCALER
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

df = pd.DataFrame({
    "age": [10, 20, 30, 40, 50]
})

scaler = MinMaxScaler()

scaled_data = scaler.fit_transform(df)

print(scaled_data)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", MinMaxScaler())
    ]
)

Outlier Handling

An outlier is a data point that is unusually far from the overall pattern of the data.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.DataFrame({
    "salary": [25000, 28000, 30000, 32000,
               35000, 38000, 500000]
})

sns.boxplot(x=df["salary"])

plt.title("Salary Outlier Detection")
plt.xlabel("Salary")
plt.show()
 

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "salary": [25000, 28000, 30000, 32000,
               35000, 38000, 500000]
})

# Calculate Q1 and Q3
Q1 = df["salary"].quantile(0.25)
Q3 = df["salary"].quantile(0.75)

# Calculate IQR
IQR = Q3 - Q1

# Calculate boundaries
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

Outlier Capping


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "salary": [25000, 28000, 30000, 32000,
               35000, 38000, 500000]
})
Q1 = df["salary"].quantile(0.25)
Q3 = df["salary"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
df["salary_capped"] = df["salary"].clip(
    lower=lower_bound,
    upper=upper_bound
)

print(df)

df["salary"].clip(lower=lower_bound, upper=upper_bound)

Log Transformation.

    Log transformation changes the scale of the values so that large numbers are compressed more than small numbers.
    
         After log transformation

              The transformed values are closer together because the large value gets compressed more strongly.

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "salary": [25000, 28000, 30000, 35000, 500000]
})
df["salary_log"] = np.log1p(df["salary"])

print(df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.histplot(df["salary"], ax=axes[0])
axes[0].set_title("Original Salary")

sns.histplot(df["salary_log"], ax=axes[1])
axes[1].set_title("Log-Transformed Salary")

plt.tight_layout()
plt.show()

In [ ]:
print("Original skewness:", df["salary"].skew())

print("Log-transformed skewness:", df["salary_log"].skew())

Z-score method for detecting outliers

What is a Z-score?
A Z-score tells us how far a value is from the average, measured in standard deviations.

In [ ]:
import pandas as pd
from scipy.stats import zscore

df = pd.DataFrame({
    "salary": [
        25000, 28000, 30000, 32000, 35000,
        38000, 29000, 31000, 34000, 30000,
        28000, 32000, 31000, 35000, 33000,
        29000, 30000, 31000, 34000, 500000
    ]
})

# Calculate Z-scores
df["z_score"] = zscore(df["salary"])

# Find potential outliers
outliers = df[df["z_score"].abs() > 3]

print(outliers)